# Cereals Data Set - Regression Analysis

## Objective
Analyze cereal ratings using regression analysis:
1. Simple linear regression (Rating vs Fiber)
2. Multiple regression (Rating vs Fiber and Sugars)

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Load cereals dataset
# The cereals dataset is from the book series and typically includes nutritional information

try:
    # Try loading from a CSV file if it exists locally
    df = pd.read_csv('cereal.csv')
    # Rename columns for consistency (if needed)
    if 'rating' in df.columns:
        df = df.rename(columns={'rating': 'Rating'})
    if 'fibre' in df.columns:
        df = df.rename(columns={'fibre': 'Fiber'})
    elif 'fiber' not in df.columns:
        # Alternative dataset structure
        pass
except FileNotFoundError:
    try:
        # Try alternate filename
        df = pd.read_csv('cereals.csv')
    except FileNotFoundError:
        print("Could not load cereal.csv or cereals.csv. Please ensure the file is in the working directory.")
        df = None
except Exception as e:
    print(f"Error loading data: {e}")
    df = None

if df is not None:
    print(f"Dataset loaded successfully with shape: {df.shape}")
    print(f"\nFirst few rows:")
    print(df.head())
    print(f"\nColumn names:")
    print(df.columns.tolist())
    print(f"\nData types:")
    print(df.dtypes)
    print(f"\nBasic statistics:")
    print(df.describe())
else:
    print("Dataset could not be loaded.")

## Part 1: Simple Linear Regression - Rating vs Fiber

In [ ]:
if df is not None:
    # Check for relevant columns
    # Look for columns containing 'rating', 'fiber', 'fibre'
    cols = df.columns.str.lower()
    rating_col = next((col for col in df.columns if col.lower() in ['rating', 'ratings']), None)
    fiber_col = next((col for col in df.columns if col.lower() in ['fiber', 'fibre']), None)
    
    if rating_col and fiber_col:
        print(f"Using columns: {rating_col} and {fiber_col}")
        
        # Remove missing values for these columns
        data = df[[rating_col, fiber_col]].dropna()
        print(f"Sample size after removing missing values: {len(data)}")
        
        # Rename for easier reference
        X = data[fiber_col].values
        y = data[rating_col].values
        
        # Fit simple linear regression using statsmodels for detailed output
        X_with_const = sm.add_constant(X)
        model_simple = sm.OLS(y, X_with_const)
        results_simple = model_simple.fit()
        
        print("\n" + "="*70)
        print("SIMPLE LINEAR REGRESSION: Rating = f(Fiber)")
        print("="*70)
        print(results_simple.summary())
        
        # Extract coefficients
        intercept = results_simple.params[0]
        slope = results_simple.params[1]
        r_squared = results_simple.rsquared
        adj_r_squared = results_simple.rsquared_adj
        rmse = np.sqrt(results_simple.mse_resid)
        
    else:
        print(f"Could not find Rating and Fiber columns. Available columns: {df.columns.tolist()}")
        print("Columns (lowercase): {}", cols.tolist())
else:
    print("Cannot proceed without data.")

In [ ]:
# Summary of Simple Regression Results
print("\n" + "="*70)
print("INTERPRETATION OF SIMPLE REGRESSION RESULTS")
print("="*70)

print(f"\na) ESTIMATED REGRESSION EQUATION:")
print(f"   Rating = {intercept:.4f} + {slope:.4f} * Fiber")
print(f"   or")
print(f"   Ŷ = {intercept:.4f} + {slope:.4f}X")

print(f"\nb) SLOPE COEFFICIENT INTERPRETATION:")
print(f"   Slope = {slope:.4f}")
print(f"   Meaning: For each additional gram of fiber per serving,")
print(f"   the predicted rating increases by {slope:.4f} points.")
print(f"   (A positive slope indicates the relationship is positive.)")

print(f"\nc) Y-INTERCEPT INTERPRETATION:")
print(f"   Y-intercept = {intercept:.4f}")
print(f"   Meaning: When fiber content is 0 grams, the predicted rating is {intercept:.4f}.")
print(f"   Practical interpretation: This may or may not make sense, as it")
print(f"   depends on whether 0 grams of fiber is a realistic scenario for cereals.")

print(f"\nd) PREDICTION ERROR / MODEL PERFORMANCE:")
print(f"   Root Mean Squared Error (RMSE) = {rmse:.4f}")
print(f"   This is the typical/average prediction error in rating points.")
print(f"   Statistic used: RMSE (Root Mean Squared Error)")
print(f"   To lower this error, we could:")
print(f"   - Add more predictor variables (multiple regression)")
print(f"   - Collect more data")
print(f"   - Transform variables if relationships are not linear")
print(f"   - Remove outliers or problematic data points")

print(f"\ne) MODEL FIT ASSESSMENT:")
print(f"   R-squared = {r_squared:.4f}")
print(f"   Adjusted R-squared = {adj_r_squared:.4f}")
print(f"   Interpretation: {r_squared*100:.2f}% of the variation in ratings")
print(f"   is explained by fiber content alone.")
print(f"   This suggests fiber explains {'a' if r_squared < 0.5 else 'a substantial'} portion")
print(f"   of the rating variation.")
print(f"   Statistic used: R-squared (Coefficient of Determination)")


In [ ]:
print(f"\nf) POINT ESTIMATE - RATING FOR 3 GRAMS OF FIBER:")
x_predict = 3
point_estimate = intercept + slope * x_predict
print(f"   Ŷ = {intercept:.4f} + {slope:.4f} * 3")
print(f"   Ŷ = {point_estimate:.4f}")
print(f"   A cereal with 3 grams of fiber is predicted to have a rating of {point_estimate:.4f}")

# Get prediction and confidence intervals
print(f"\ng) 95% CONFIDENCE INTERVAL FOR MEAN RATING (Fiber = 3 grams):")
prediction = results_simple.get_prediction([1.0, x_predict])
pred_summary = prediction.summary_frame(alpha=0.05)
ci_lower = pred_summary['mean_ci_lower'][0]
ci_upper = pred_summary['mean_ci_upper'][0]
print(f"   95% CI for mean rating: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"   Interpretation: We are 95% confident that the true average rating")
print(f"   for all cereals with 3 grams of fiber lies between {ci_lower:.4f} and {ci_upper:.4f}")

print(f"\nh) 95% PREDICTION INTERVAL FOR INDIVIDUAL RATING (Fiber = 3 grams):")
pred_ci_lower = pred_summary['obs_ci_lower'][0]
pred_ci_upper = pred_summary['obs_ci_upper'][0]
print(f"   95% PI for individual rating: [{pred_ci_lower:.4f}, {pred_ci_upper:.4f}]")
print(f"   Interpretation: We are 95% confident that a randomly chosen cereal")
print(f"   with 3 grams of fiber will have a rating between {pred_ci_lower:.4f} and {pred_ci_upper:.4f}")
print(f"   Note: PI is wider than CI because there's more uncertainty in predicting")
print(f"   individual values than average values.")

print(f"\ni) EXPECTED SCATTER PLOT APPEARANCE:")
print(f"   Based on positive slope ({slope:.4f}) and R² = {r_squared:.4f}:")
print(f"   - The scatter plot should show a generally positive linear trend")
print(f"   - Points should cluster around the regression line")
print(f"   - The tightness of clustering depends on R² value")
print(f"   - {'Strong' if r_squared > 0.7 else 'Moderate' if r_squared > 0.4 else 'Weak'} linear relationship expected")

In [ ]:
# Visualize the simple linear regression
if df is not None and rating_col and fiber_col:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Scatter plot with regression line
    ax = axes[0, 0]
    ax.scatter(X, y, alpha=0.6, edgecolors='k', linewidth=0.5)
    x_line = np.linspace(X.min(), X.max(), 100)
    y_line = intercept + slope * x_line
    ax.plot(x_line, y_line, 'r-', linewidth=2, label='Regression Line')
    ax.axvline(x=3, color='g', linestyle='--', alpha=0.7, label='x=3 (for prediction)')
    ax.axhline(y=point_estimate, color='g', linestyle='--', alpha=0.7)
    ax.plot(3, point_estimate, 'go', markersize=10, label=f'Prediction at x=3')
    ax.set_xlabel(fiber_col, fontsize=11)
    ax.set_ylabel(rating_col, fontsize=11)
    ax.set_title('Scatter Plot with Regression Line', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Residuals
    ax = axes[0, 1]
    fitted_vals = results_simple.fittedvalues
    residuals = results_simple.resid
    ax.scatter(fitted_vals, residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
    ax.axhline(y=0, color='r', linestyle='--', linewidth=2)
    ax.set_xlabel('Fitted Values', fontsize=11)
    ax.set_ylabel('Residuals', fontsize=11)
    ax.set_title('Residual Plot (Check for patterns)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Q-Q plot for normality of residuals
    ax = axes[1, 0]
    sm.qqplot(residuals, line='45', ax=ax)
    ax.set_title('Q-Q Plot (Check Normality of Residuals)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Plot 4: Distribution of residuals
    ax = axes[1, 1]
    ax.hist(residuals, bins=15, edgecolor='k', alpha=0.7)
    ax.set_xlabel('Residuals', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Distribution of Residuals', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('simple_regression_diagnostics.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("\nDiagnostic plots created successfully!")

## Part 2: Multiple Linear Regression - Rating vs Fiber and Sugars

In [ ]:
if df is not None and rating_col and fiber_col:
    # Look for sugars column
    sugar_col = next((col for col in df.columns if col.lower() in ['sugar', 'sugars']), None)
    
    if sugar_col:
        print(f"Using columns: {rating_col}, {fiber_col}, and {sugar_col}")
        
        # Prepare data
        data_multi = df[[rating_col, fiber_col, sugar_col]].dropna()
        print(f"Sample size after removing missing values: {len(data_multi)}")
        
        # Extract variables
        y_multi = data_multi[rating_col].values
        X_fiber_multi = data_multi[fiber_col].values
        X_sugar_multi = data_multi[sugar_col].values
        
        # Fit multiple regression
        X_matrix = sm.add_constant(np.column_stack([X_fiber_multi, X_sugar_multi]))
        model_multi = sm.OLS(y_multi, X_matrix)
        results_multi = model_multi.fit()
        
        print("\n" + "="*70)
        print("MULTIPLE LINEAR REGRESSION: Rating = f(Fiber, Sugars)")
        print("="*70)
        print(results_multi.summary())
        
        # Extract coefficients
        b0 = results_multi.params[0]
        b1 = results_multi.params[1]  # Fiber coefficient
        b2 = results_multi.params[2]  # Sugars coefficient
        r2_multi = results_multi.rsquared
        adj_r2_multi = results_multi.rsquared_adj
        rmse_multi = np.sqrt(results_multi.mse_resid)
        
    else:
        print(f"Sugar column not found. Available columns: {df.columns.tolist()}")

In [ ]:
if df is not None and 'results_multi' in dir():
    print("\n" + "="*70)
    print("INTERPRETATION OF MULTIPLE REGRESSION RESULTS")
    print("="*70)
    
    print(f"\na) ESTIMATED REGRESSION EQUATION:")
    print(f"   Rating = {b0:.4f} + {b1:.4f} * Fiber + {b2:.4f} * Sugars")
    print(f"   or")
    print(f"   Ŷ = {b0:.4f} + {b1:.4f}X₁ + {b2:.4f}X₂")
    print(f"   where X₁ = Fiber, X₂ = Sugars")
    
    print(f"\nb) FIBER COEFFICIENT INTERPRETATION:")
    print(f"   Fiber coefficient (b₁) = {b1:.4f}")
    print(f"   Meaning: Holding sugars constant (controlling for sugars),")
    print(f"   for each additional gram of fiber, the predicted rating")
    print(f"   {'increases' if b1 > 0 else 'decreases'} by {abs(b1):.4f} points.")
    print(f"   This is the PARTIAL effect of fiber, adjusting for sugars.")
    
    print(f"\nc) COMPARISON OF R² VALUES:")
    print(f"   Simple regression (Fiber only):")
    print(f"      R² = {r_squared:.4f} ({r_squared*100:.2f}%)")
    print(f"   Multiple regression (Fiber + Sugars):")
    print(f"      R² = {r2_multi:.4f} ({r2_multi*100:.2f}%)")
    print(f"   Improvement: {(r2_multi - r_squared)*100:.2f} percentage points")
    print(f"   Conclusion: Adding sugars as a predictor")
    print(f"   {'improves' if r2_multi > r_squared else 'does not improve'} model fit.")
    
    print(f"\n   Adjusted R² comparison:")
    print(f"      Simple: {adj_r_squared:.4f}")
    print(f"      Multiple: {adj_r2_multi:.4f}")
    print(f"   Adjusted R² accounts for the number of predictors.")
    
    print(f"\nd) MODEL PERFORMANCE COMPARISON:")
    print(f"   Simple regression RMSE: {rmse:.4f}")
    print(f"   Multiple regression RMSE: {rmse_multi:.4f}")
    print(f"   Change: {rmse_multi - rmse:.4f}")
    print(f"   Interpretation: {'Lower RMSE' if rmse_multi < rmse else 'Higher RMSE'} indicates {'better' if rmse_multi < rmse else 'worse'} predictions")
    
    print(f"\ne) SUMMARY:")
    print(f"   The multiple regression model with both fiber and sugars")
    print(f"   {'explains more' if r2_multi > r_squared else 'explains less'} variation than fiber alone.")
    print(f"   Both predictors together provide a {'better' if r2_multi > r_squared else 'worse or similar'} fit.")

In [ ]:
if df is not None and 'results_multi' in dir():
    # Create comparison table
    print("\n" + "="*70)
    print("DETAILED COMPARISON: SIMPLE vs MULTIPLE REGRESSION")
    print("="*70)
    
    comparison_data = {
        'Metric': ['R-squared', 'Adjusted R²', 'RMSE', 'AIC', 'BIC', 'Observations'],
        'Simple (Fiber)': [
            f'{r_squared:.4f}',
            f'{adj_r_squared:.4f}',
            f'{rmse:.4f}',
            f'{results_simple.aic:.2f}',
            f'{results_simple.bic:.2f}',
            f'{len(data)}'
        ],
        'Multiple (Fiber + Sugars)': [
            f'{r2_multi:.4f}',
            f'{adj_r2_multi:.4f}',
            f'{rmse_multi:.4f}',
            f'{results_multi.aic:.2f}',
            f'{results_multi.bic:.2f}',
            f'{len(data_multi)}'
        ]
    }
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\n", comparison_df.to_string(index=False))
    
    # Coefficient comparison
    print("\n" + "-"*70)
    print("COEFFICIENT COMPARISON")
    print("-"*70)
    print(f"\nFiber coefficient:")
    print(f"  Simple regression:   {slope:.4f}")
    print(f"  Multiple regression: {b1:.4f}")
    print(f"  Change: {b1 - slope:.4f}")
    print(f"  The fiber effect {'increased' if b1 > slope else 'decreased'} when sugars were added to the model.")
    print(f"  This suggests fiber and sugars {'are' if abs(b1 - slope) > 0.01 else 'are not'} related.")
    
    print(f"\nSugars coefficient:")
    print(f"  Simple regression:   (not included)")
    print(f"  Multiple regression: {b2:.4f}")
    print(f"  Interpretation: Holding fiber constant, each additional gram")
    print(f"                  of sugar changes rating by {b2:.4f} points.")

In [ ]:
if df is not None and 'results_multi' in dir():
    # Additional Analysis: Correlation matrix and VIF
    print("\n" + "="*70)
    print("ADDITIONAL DIAGNOSTICS FOR MULTIPLE REGRESSION")
    print("="*70)
    
    # Correlation matrix
    print("\nCORRELATION MATRIX:")
    corr_matrix = data_multi.corr()
    print(corr_matrix)
    
    # Variance Inflation Factor (VIF)
    print("\nVARIANCE INFLATION FACTORS (VIF):")
    print("(Values > 10 indicate potential multicollinearity)")
    vif_data = pd.DataFrame()
    vif_data["Variable"] = [fiber_col, sugar_col]
    vif_data["VIF"] = [
        variance_inflation_factor(np.column_stack([X_fiber_multi, X_sugar_multi]), i) 
        for i in range(2)
    ]
    print(vif_data)
    
    # F-test for overall model significance
    print(f"\nOVERALL MODEL F-TEST:")
    print(f"  F-statistic: {results_multi.fvalue:.4f}")
    print(f"  P-value: {results_multi.f_pvalue:.4e}")
    print(f"  Interpretation: The model is {'statistically significant' if results_multi.f_pvalue < 0.05 else 'not statistically significant'} at α=0.05")

In [ ]:
if df is not None and 'results_multi' in dir():
    # Visualizations for multiple regression
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Fiber vs Rating (with colors for sugar levels)
    ax = axes[0, 0]
    scatter = ax.scatter(X_fiber_multi, y_multi, c=X_sugar_multi, cmap='viridis', 
                         alpha=0.6, s=80, edgecolors='k', linewidth=0.5)
    ax.set_xlabel(fiber_col, fontsize=11)
    ax.set_ylabel(rating_col, fontsize=11)
    ax.set_title(f'{rating_col} vs {fiber_col}\n(colored by {sugar_col})', 
                 fontsize=12, fontweight='bold')
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label(sugar_col, fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Sugars vs Rating (with colors for fiber levels)
    ax = axes[0, 1]
    scatter2 = ax.scatter(X_sugar_multi, y_multi, c=X_fiber_multi, cmap='plasma', 
                          alpha=0.6, s=80, edgecolors='k', linewidth=0.5)
    ax.set_xlabel(sugar_col, fontsize=11)
    ax.set_ylabel(rating_col, fontsize=11)
    ax.set_title(f'{rating_col} vs {sugar_col}\n(colored by {fiber_col})', 
                 fontsize=12, fontweight='bold')
    cbar2 = plt.colorbar(scatter2, ax=ax)
    cbar2.set_label(fiber_col, fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Fitted vs Actual for Multiple Regression
    ax = axes[1, 0]
    fitted_multi = results_multi.fittedvalues
    ax.scatter(fitted_multi, y_multi, alpha=0.6, edgecolors='k', linewidth=0.5)
    ax.plot([y_multi.min(), y_multi.max()], [y_multi.min(), y_multi.max()], 
            'r--', linewidth=2, label='Perfect fit')
    ax.set_xlabel('Fitted Values', fontsize=11)
    ax.set_ylabel('Actual Values', fontsize=11)
    ax.set_title(f'Actual vs Fitted (Multiple R² = {r2_multi:.4f})', 
                 fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 4: Model Comparison - R² and RMSE
    ax = axes[1, 1]
    models = ['Simple\n(Fiber)', 'Multiple\n(Fiber + Sugars)']
    r2_values = [r_squared, r2_multi]
    colors_bar = ['#1f77b4', '#ff7f0e']
    
    x_pos = np.arange(len(models))
    ax.bar(x_pos - 0.2, r2_values, 0.4, label='R²', color=colors_bar, alpha=0.7, edgecolor='k')
    ax.set_ylabel('R² Value', fontsize=11)
    ax.set_title('Model Comparison: R² Values', fontsize=12, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(models)
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(r2_values):
        ax.text(i - 0.2, v + 0.02, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('multiple_regression_analysis.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("\nMultiple regression visualization created successfully!")

## Summary and Recommendations

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY AND CONCLUSIONS")
print("="*70)

if df is not None and 'results_simple' in dir() and 'results_multi' in dir():
    print("""
KEY FINDINGS:

1. SIMPLE REGRESSION (Fiber only):
   - Captures initial relationship between fiber and rating
   - R² indicates how much variation fiber alone explains
   - RMSE shows typical prediction error from this model
   
2. MULTIPLE REGRESSION (Fiber + Sugars):
   - Additional predictor (sugars) helps explain more variation
   - Fiber coefficient changes when sugars are added
   - This change reveals the partial effect of fiber
   
3. STATISTICAL SIGNIFICANCE:
   - Check t-statistics and p-values in the regression output
   - Variables with p < 0.05 are significant at 5% level
   - F-test evaluates overall model significance
   
4. MODEL SELECTION CONSIDERATIONS:
   - Compare R² between models
   - Consider adjusted R² (accounts for number of variables)
   - Check diagnostic plots for assumptions
   - Examine residuals for patterns and violations
   
5. PRACTICAL IMPLICATIONS:
   - Confidence intervals show precision of estimates
   - Prediction intervals account for individual variation
   - VIF values indicate multicollinearity issues
   - Correlation matrix shows relationships between predictors
   
6. RECOMMENDATIONS:
   - Consider adding more meaningful predictors
   - Check for interaction effects between fiber and sugars
   - Validate model with out-of-sample predictions
   - Consider non-linear relationships if residual plots suggest
   - Investigate outliers and influential observations
   
7. INTERPRETATION BEST PRACTICES:
   - Always interpret coefficients in context
   - Report both statistical and practical significance
   - Use confidence/prediction intervals for uncertainty
   - Check all regression assumptions before interpreting results
""")

else:
    print("\nPlease run the regression analyses above to see the complete summary.")

print("\n" + "="*70)
print("END OF ANALYSIS")
print("="*70)

## Part 3: Data Mining Concepts and Theory Questions

In [ ]:
print("\n" + "="*70)
print("DATA MINING CONCEPTS: THEORY QUESTIONS AND ANSWERS")
print("="*70)

print("""
a. SUPERVISED vs UNSUPERVISED METHODS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SUPERVISED LEARNING:
  - Uses labeled data (target variable is known)
  - Goal: Learn mapping from inputs to outputs
  - Training: Model learns relationship between features and target
  
  Associated Tasks:
  ✓ Classification (predict categorical outcomes)
  ✓ Regression (predict continuous outcomes)
  ✓ Fraud detection (yes/no fraud)
  ✓ Credit rating prediction
  ✓ Disease diagnosis

UNSUPERVISED LEARNING:
  - Uses unlabeled data (no target variable)
  - Goal: Discover hidden patterns or structure
  - Training: Model finds natural groupings or associations
  
  Associated Tasks:
  ✓ Clustering (group similar observations)
  ✓ Dimensionality reduction (reduce number of features)
  ✓ Association rule mining (find relationships between items)
  ✓ Anomaly detection (find unusual observations)
  ✓ Market segmentation

BOTH SUPERVISED AND UNSUPERVISED:
  ✓ Anomaly detection (can use labeled or unlabeled data)
  ✓ Text mining (can be supervised for sentiment analysis
    or unsupervised for topic modeling)
  ✓ Feature engineering (often uses both approaches)
  ✓ Semi-supervised learning (uses both labeled & unlabeled data)


b. TRAINING SET, TEST SET, AND VALIDATION SET
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

TRAINING SET:
  - Subset of data used to fit/train the model
  - Model learns patterns from this data
  - Typical size: 60-70% of total data
  - Purpose: Adjust model parameters/weights

TEST SET:
  - Completely separate data used for final evaluation
  - Unseen by the model during training
  - Typical size: 15-20% of total data
  - Purpose: Estimate real-world model performance
  - Used only ONCE at the end
  - Simulates new, unseen data

VALIDATION SET (also called Development Set):
  - Separate from training but may be used multiple times
  - Typical size: 15-20% of total data
  - Purpose: 
    • Tune hyperparameters (learning rate, regularization)
    • Decide when to stop training
    • Select best model from multiple candidates
  - Used during model development

SPLITTING STRATEGY:
  Total Data → 60% Training | 20% Validation | 20% Test

SHOULD WE MAXIMIZE TRAINING ACCURACY?
  NO! Here's why:
  ✗ High training accuracy often indicates OVERFITTING
  ✗ Model memorizes training data instead of learning patterns
  ✗ Poor performance on new, unseen data
  ✓ Better goal: Good training accuracy + Good validation accuracy
  
WHAT ABOUT VALIDATION ACCURACY?
  YES, we should strive for high validation accuracy because:
  ✓ Validation set hasn't been used to optimize model
  ✓ Better indicator of real-world performance
  ✓ Helps identify overfitting/underfitting
  ✓ Used to tune hyperparameters fairly
  NOTE: Final assessment uses TEST set (the true unseen data)


c. BIAS–VARIANCE TRADE-OFF, OVERFITTING, AND UNDERFITTING
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BIAS (Error from wrong assumptions):
  - Measures systematic error
  - HIGH BIAS = Model oversimplifies (misses true patterns)
  - Associated with UNDERFITTING
  
VARIANCE (Sensitivity to training data):
  - Measures how much model prediction changes with different training data
  - HIGH VARIANCE = Model memorizes noise and fluctuations
  - Associated with OVERFITTING

INTERPRETATIONS:

  HIGH BIAS → UNDERFITTING:
    • Model too simple for the problem
    • Poor performance on both training AND test sets
    • Example: Linear regression for non-linear relationship
    • Problem: Not capturing complexity of data
    • Solution: More complex model, add features, train longer

  HIGH VARIANCE → OVERFITTING:
    • Model too complex for the problem
    • Excellent performance on training set
    • Poor performance on test/validation sets
    • Example: 10th-degree polynomial for simple linear trend
    • Problem: Model fits noise, doesn't generalize
    • Solution: Simpler model, more data, regularization, dropout

BIAS-VARIANCE TRADE-OFF:
    Decreasing bias → increase variance
    Decreasing variance → increase bias
    Goal: Find optimal balance for best test performance


d. WHY BALANCE DATA?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Problem: Class Imbalance
  - Dataset has unequal distribution of classes
  - Example: 99% normal, 1% fraud
  
Why it's problematic:
  ✗ Model biased toward majority class
  ✗ Ignores minority class (the important one!)
  ✗ High accuracy, but fails to identify fraud/disease/etc.
  ✗ Cost of misclassification is often asymmetric

Example of useless model:
  Fraud data: 10,000 normal, 100 frauds
  "Smart" model predicts "always normal"
  Accuracy: 99% (but useless - catches 0 frauds!)

Solutions for class imbalance:
  1. OVERSAMPLING: Duplicate minority class samples
  2. UNDERSAMPLING: Remove some majority class samples
  3. STRATIFIED SAMPLING: Maintain class proportions in subsets
  4. SMOTE: Synthetically create new minority samples
  5. COST-BASED LEARNING: Assign higher cost to misclassifying minority
  6. ENSEMBLE METHODS: Combine multiple models


e. FRAUD CLASSIFICATION PROBLEM
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Given Information:
  • Training set total: 10,000 records
  • Fraudulent records: 400
  • Desired proportion of fraudulent: 20%
  • Current proportion: 400/10,000 = 4%

Goal: Find how many fraudulent records to resample

METHOD 1: Using proportion formula
  Let n = additional fraudulent records needed
  
  (400 + n) / 10,000 = 0.20
  400 + n = 2000
  n = 1600
  
  We need to RESAMPLE 1,600 fraudulent records
  (through oversampling/duplication, this means replacing with 1600
  samples drawn from the 400 original frauds)

VERIFICATION:
  Original frauds: 400
  Additional resampled frauds: 1,600
  New frauds in balanced dataset: 2,400
  
  Wait, let me reconsider the question. It asks "how many need to be 
  resampled" - this likely means how many to add through resampling.
  
  Current: 400 fraudulent out of 10,000 = 4%
  Desired: 20% fraudulent
  
  If we keep the 10,000 total:
    10,000 × 0.20 = 2,000 fraudulent needed
    Additional: 2,000 - 400 = 1,600 fraudulent records
  
  If we use oversampling (replicate existing 400):
    Need to replicate 400 records 4 times (400 × 5 = 2,000)
    Actually, need 1,600 additional from the 400 pool
    This is sampling with replacement from the 400 frauds
  
  ANSWER: 1,600 fraudulent records need to be resampled
  
  Result: New balanced dataset
    - 8,000 normal records (unchanged)
    - 2,400 fraudulent records (400 original + 1,600 resampled)
    - Wait, that's 10,400 total...
  
  Let me reconsider. If we want exactly 20% in a balanced set:
    Option A: Keep total at 10,000
      Normal: 8,000 (80%)
      Fraudulent: 2,000 (20%)
      Resampled frauds needed: 2,000 - 400 = 1,600
      
    Option B: Grow the dataset
      Keep all 10,000 - 400 = 9,600 normal
      Resample frauds to match: 9,600 × (20/80) = 2,400
      Additional frauds: 2,400 - 400 = 2,000

  Most common interpretation: We want 20% in the BALANCED dataset
  If we keep all normal records: 9,600 normal, need 2,400 fraud
  Additional: 2,400 - 400 = 2,000
  
  Let me use the cleaner interpretation:
  New dataset should have 20% frauds:
    If we keep 400 originals as base normal operations
    We need frauds = 0.20 × total
    
  Actually, the simplest: "proportion of fraudulent in balanced dataset = 20%"
  means after balancing: fraud% = 20%, normal% = 80%
  
  If we resample to make balanced (equal classes):
    50% fraud, 50% normal
    We'd need 10,000 fraud (if 10,000 normal)
    Additional: 10,000 - 400 = 9,600
    
  But they want 20% not 50%, so:
  20% fraud means 80% normal
  Ratio: fraud:normal = 20:80 = 1:4
  
  If 400 remain as normal: 400/(n+400) should NOT be fraud
  If fraud is 20%: fraud/(fraud+normal) = 0.20
  
  Common approach: Keep all records, resample minority
  Normal: 10,000 - 400 = 9,600
  Frauds: need 20% of total
  0.20 = fraud / (fraud + 9,600)
  0.20(fraud + 9,600) = fraud
  0.20*fraud + 1,920 = fraud
  1,920 = 0.80*fraud
  fraud = 2,400
  
  Additional resampled: 2,400 - 400 = 2,000
  
  FINAL ANSWER: 2,000 fraudulent records need to be resampled
  
  Verification:
    Normal records: 9,600
    Fraudulent (original: 400 + resampled: 2,000): 2,400
    Total: 12,000
    Fraudulent proportion: 2,400/12,000 = 20% ✓
""")

# Calculate and display the fraud balancing example
print("\n" + "="*70)
print("FRAUD CLASSIFICATION CALCULATION - DETAILED BREAKDOWN")
print("="*70)

original_total = 10000
fraudulent_records = 400
normal_records = original_total - fraudulent_records
desired_fraud_pct = 0.20  # 20%

print(f"\nOriginal Dataset:")
print(f"  Total records: {original_total}")
print(f"  Fraudulent: {fraudulent_records}")
print(f"  Normal: {normal_records}")
print(f"  Current proportion fraudulent: {fraudulent_records/original_total:.2%}")

# Calculate needed fraudulent records for 20%
# If normal stays at 9,600 and we want fraud% = 20%:
# fraud / (fraud + 9600) = 0.20
# fraud = 0.20 * (fraud + 9600)
# fraud = 0.20*fraud + 1920
# 0.80*fraud = 1920
# fraud = 2400

fraudulent_needed = normal_records * (desired_fraud_pct / (1 - desired_fraud_pct))
resampled_needed = int(fraudulent_needed - fraudulent_records)

print(f"\nTarget: 20% fraudulent in balanced dataset")
print(f"  Normal records (unchanged): {normal_records}")
print(f"  Fraudulent records needed: {int(fraudulent_needed)}")
print(f"    - Original: {fraudulent_records}")
print(f"    - Additional (resampled): {resampled_needed}")

total_after_balancing = normal_records + int(fraudulent_needed)
final_fraud_pct = int(fraudulent_needed) / total_after_balancing

print(f"\nBalanced Dataset:")
print(f"  Total records: {total_after_balancing}")
print(f"  Fraudulent: {int(fraudulent_needed)}")
print(f"  Normal: {normal_records}")
print(f"  Proportion fraudulent: {final_fraud_pct:.2%}")

print(f"\n✓ ANSWER: {resampled_needed} fraudulent records need to be resampled")
print("="*70)